## Tiny NeRF — depth-guided training on `transforms_extended.json`

Requires a `transforms_extended.json` produced by `images_generator.py` (contains
`depth_path` and `mask_path` per frame).  All model/training/render utilities live
in `nerf_module.py` — this notebook is a thin driver.

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
from IPython import display as ipython_display

# All NeRF utilities — model, encoding, sampling, rendering, dataset, train
from nerf_module import (
    NerfConfig, TinyNerfModel, VeryTinyNerfModel,
    positional_encoding, get_ray_bundle,
    compute_query_points_from_rays, compute_query_points_from_depth,
    render_volume_density, run_one_iter, render_image,
    query_radiance, get_minibatches,
    save_checkpoint, load_checkpoint,
    NerfDataset, train,
)

## Config

In [ ]:
# ── Paths ──────────────────────────────────────────────────────────────────
TRANSFORMS_JSON_PATH = "./output/sworshield_render_nerf/transforms_extended.json"
OUTPUT_DIR           = "./output/sworshield_render_nerf"   # previews + loss curve go here
MODEL_CACHE_PATH     = "./output/sworshield_render_nerf/model/tinynerf_model_cache.pkl"

# Texture bake (runs once at the end)
ENABLE_TEXTURE_BAKE    = True
POSITIONAL_MAP_EXR_PATH = "./output/ium/inverse_uv_mapping.exr"
OUTPUT_TEXTURE_PATH    = "./output/ium/nerf_baked_texture.png"
OUTPUT_CONFIDENCE_PATH = "./output/ium/nerf_baked_confidence.png"

# ── NeRF hyper-parameters ──────────────────────────────────────────────────
# near/far in world units (apply_scale=False → fg depth ≈ 9.2–10.4 for this scene)
cfg = NerfConfig(
    num_encoding_functions = 6,
    filter_size            = 128,
    near                   = 7.0,
    far                    = 13.0,
    depth_window           = 0.15,   # ±window around per-pixel depth
    depth_samples_per_ray  = 8,
    chunk_size             = 16384,
)

# ── Training hyper-parameters ──────────────────────────────────────────────
NUM_ITERS     = 10000
BATCH_SIZE    = 4096
MASK_BIAS     = 0.9     # fraction of each batch sampled from foreground pixels
LR            = 5e-3
DISPLAY_EVERY = 100
SEED          = 9458

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', device)

## Load dataset

In [ ]:
# NerfDataset validates that depth_path + mask_path are present per frame,
# pre-computes all ray bundles (CPU), and builds foreground/background index pools.
dataset = NerfDataset(TRANSFORMS_JSON_PATH, device=device)

# Convenience aliases used by the bake cell below
height       = dataset.H
width        = dataset.W
focal_length = torch.tensor([dataset.focal], dtype=torch.float32, device=device)

#### Display test image

In [ ]:
testimg, testpose, testdepth = dataset.get_test_frame()
plt.figure(figsize=(8, 5))
plt.imshow(testimg.cpu().numpy())
plt.title(f'Test frame (idx={dataset.test_idx})')
plt.axis('off')
plt.show()

## Train TinyNeRF!

Uses **random-ray batching** with depth-guided sampling for foreground pixels
(`mask_bias` fraction) and uniform sampling for background pixels.
Every `DISPLAY_EVERY` iterations the loss + PSNR curves update inline and a
preview of the test frame is saved to `<OUTPUT_DIR>/nerf_train/`.
If a checkpoint exists the training resumes from where it left off.

In [ ]:
# -- Inline monitoring callback ------------------------------------------
_fig, (_ax_loss, _ax_psnr) = plt.subplots(1, 2, figsize=(12, 4))

def on_step(iter_num, loss, psnrs_list, losses_list, mdl):
    ipython_display.clear_output(wait=True)
    _ax_loss.cla(); _ax_psnr.cla()
    iters = list(range(0, iter_num + 1, DISPLAY_EVERY))[:len(losses_list)]
    _ax_loss.semilogy(iters, losses_list, color='tab:blue')
    _ax_loss.set_title('Loss'); _ax_loss.set_xlabel('Iter')
    _ax_psnr.plot(iters, psnrs_list, color='tab:orange')
    _ax_psnr.set_title('PSNR (dB)'); _ax_psnr.set_xlabel('Iter')
    _fig.tight_layout()
    plt.show()
    print(f'  [{iter_num:5d}] loss={loss:.5f}  PSNR={psnrs_list[-1]:.2f} dB')

# -- Train ---------------------------------------------------------------
model = train(
    dataset,
    cfg,
    num_iters     = NUM_ITERS,
    batch_size    = BATCH_SIZE,
    mask_bias     = MASK_BIAS,
    lr            = LR,
    ckpt_path     = MODEL_CACHE_PATH,
    display_every = DISPLAY_EVERY,
    output_dir    = OUTPUT_DIR,
    on_step       = on_step,
    seed          = SEED,
)
print('Training done.')

## Texture bake from IUM position map

Projects every valid texel of the IUM position map into all training cameras,
queries the trained NeRF for the color, and averages contributions.

In [ ]:
import importlib

def _resolve(path):
    cwd = os.getcwd()
    for c in [path, os.path.join(cwd, path), os.path.join(cwd, path.lstrip('./\\'))]:
        c = os.path.normpath(c)
        if os.path.exists(c):
            return c
    return os.path.normpath(path)


def load_exr_position_map_rgba(exr_path):
    OpenEXR = importlib.import_module('OpenEXR')
    Imath   = importlib.import_module('Imath')
    f  = OpenEXR.InputFile(exr_path)
    dw = f.header()['dataWindow']
    w  = dw.max.x - dw.min.x + 1
    h  = dw.max.y - dw.min.y + 1
    PT = Imath.PixelType(Imath.PixelType.FLOAT)
    chs = list(f.header()['channels'].keys())
    def _ch(*names):
        for n in names:
            if n in chs:
                return np.frombuffer(f.channel(n, PT), dtype=np.float32).reshape(h, w)
        return None
    x = _ch('R', 'X'); y = _ch('G', 'Y'); z = _ch('B', 'Z'); a = _ch('A', 'alpha')
    if x is None or y is None or z is None:
        raise RuntimeError(f'Position map missing RGB/XYZ channels; found: {chs}')
    if a is None:
        a = np.ones((h, w), dtype=np.float32)
    return np.stack([x, y, z], axis=-1).astype(np.float32), a.astype(np.float32)


def query_nerf_for_points(points_world, cam2world_batch, model, cfg):
    """Query NeRF color for N world points observed from C cameras.
    points_world: (N,3)  cam2world_batch: (C,4,4)  returns (N*C, 3) colors.
    """
    C = cam2world_batch.shape[0]
    N = points_world.shape[0]
    cam_pos   = cam2world_batch[:, :3, 3]          # (C,3)
    # For each (point, camera) pair: origin = cam_pos, dir = point - cam_pos
    origins   = cam_pos.unsqueeze(1).expand(C, N, 3).reshape(-1, 3)  # (C*N,3)
    ray_dirs  = (points_world.unsqueeze(0) - cam_pos.unsqueeze(1)).reshape(-1, 3)  # (C*N,3)
    t_hits    = torch.linalg.norm(ray_dirs, dim=-1).clamp_min(1e-8)  # (C*N,)
    ray_dirs_n = ray_dirs / t_hits.unsqueeze(-1)
    with torch.no_grad():
        rgb, _, _ = run_one_iter(origins, ray_dirs_n, model, cfg,
                                  target_depth=t_hits, randomize=False)
    return rgb  # (C*N, 3)


def bake_texture_from_position_map(pos_map_path, out_tex_path, out_conf_path,
                                    model, dataset, cfg):
    resolved = _resolve(pos_map_path)
    if not os.path.exists(resolved):
        raise FileNotFoundError(f'Position map not found: {pos_map_path}')

    print(f'[Bake] Loading position map: {resolved}')
    pos_np, msk_np = load_exr_position_map_rgba(resolved)
    h_tex, w_tex   = pos_np.shape[:2]

    valid_np  = np.isfinite(pos_np).all(axis=-1) & (msk_np > 0.5)
    valid_idx = np.argwhere(valid_np)  # (K,2) yx
    if valid_idx.shape[0] == 0:
        raise RuntimeError('No valid texels in position map')

    dev   = dataset.device
    f_val = dataset.focal
    H, W  = dataset.H, dataset.W

    points = torch.from_numpy(pos_np[valid_np]).to(dev)  # (K,3)
    K      = points.shape[0]

    cam_poses_list = [dataset.get_frame(i)[1] for i in range(dataset.num_frames)]
    cam_poses = torch.stack(cam_poses_list)  # (C,4,4)

    # Project points to all cameras, keep visible pairs
    world2cam = torch.linalg.inv(cam_poses)  # (C,4,4)
    pts_h     = torch.cat([points, torch.ones(K, 1, device=dev)], dim=-1)  # (K,4)
    p_cam     = torch.einsum('cij,nj->cni', world2cam, pts_h)               # (C,K,4)
    z_c       = p_cam[..., 2]
    depth_cam = -z_c
    u         = f_val * (p_cam[..., 0] / depth_cam.clamp_min(1e-8)) + W * 0.5
    v         = -f_val * (p_cam[..., 1] / depth_cam.clamp_min(1e-8)) + H * 0.5
    visible   = (z_c < -1e-6) & (u >= 0) & (u < W) & (v >= 0) & (v < H)

    cam_ids, tex_ids = torch.nonzero(visible, as_tuple=True)
    if tex_ids.numel() == 0:
        raise RuntimeError('No texel-camera pairs visible')

    print(f'[Bake] Valid texels: {K} | Visible pairs: {tex_ids.numel()}')

    pair_points = points[tex_ids]          # (M,3)
    pair_cams   = cam_poses[cam_ids]       # (M,4,4)
    cam_pos     = pair_cams[:, :3, 3]      # (M,3)
    ray_dirs    = pair_points - cam_pos    # (M,3)
    t_hits      = torch.linalg.norm(ray_dirs, dim=-1).clamp_min(1e-8)
    ray_dirs_n  = ray_dirs / t_hits.unsqueeze(-1)

    model.eval()
    with torch.no_grad():
        colors, _, _ = run_one_iter(cam_pos, ray_dirs_n, model, cfg,
                                     target_depth=t_hits, randomize=False)

    # Average per texel
    sum_rgb = torch.zeros(K, 3, device=dev).index_add_(0, tex_ids, colors)
    count   = torch.zeros(K, device=dev).index_add_(0, tex_ids,
                  torch.ones(tex_ids.shape[0], device=dev))
    mean_rgb = sum_rgb / count.clamp_min(1.0).unsqueeze(-1)

    # Confidence from spread
    dev_col   = torch.linalg.norm(colors - mean_rgb[tex_ids], dim=-1)
    spread_sum = torch.zeros(K, device=dev).index_add_(0, tex_ids, dev_col)
    spread     = spread_sum / count.clamp_min(1.0)
    conf       = torch.exp(-8.0 * spread).clamp(0, 1)
    conf       = torch.where(count > 0, conf, torch.zeros_like(conf))

    texture    = torch.zeros(h_tex, w_tex, 3, device=dev)
    confidence = torch.zeros(h_tex, w_tex,    device=dev)
    vy = torch.from_numpy(valid_idx[:, 0]).to(dev)
    vx = torch.from_numpy(valid_idx[:, 1]).to(dev)
    texture[vy, vx]    = mean_rgb.clamp(0, 1)
    confidence[vy, vx] = conf

    tex_np  = texture.cpu().numpy()
    conf_np = confidence.cpu().numpy()
    valid_m = conf_np > 0
    if np.any(valid_m):
        tex_np[~valid_m] = np.mean(tex_np[valid_m], axis=0)

    os.makedirs(os.path.dirname(os.path.normpath(out_tex_path)),  exist_ok=True)
    os.makedirs(os.path.dirname(os.path.normpath(out_conf_path)), exist_ok=True)
    Image.fromarray((np.clip(tex_np,  0, 1) * 255).astype(np.uint8)).save(out_tex_path)
    Image.fromarray((np.clip(conf_np, 0, 1) * 255).astype(np.uint8), mode='L').save(out_conf_path)
    print(f'[Bake] Texture  → {out_tex_path}')
    print(f'[Bake] Confidence → {out_conf_path}')


if ENABLE_TEXTURE_BAKE:
    try:
        bake_texture_from_position_map(
            POSITIONAL_MAP_EXR_PATH,
            OUTPUT_TEXTURE_PATH,
            OUTPUT_CONFIDENCE_PATH,
            model, dataset, cfg,
        )
    except Exception as e:
        print(f'[Bake] Error: {e}')

print('Done!')